In [1]:
import warnings
warnings.filterwarnings("ignore")

from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import pandas_ta as ta
import yfinance as yf
from backtesting import Backtest, Strategy

In [82]:
SYMBOL           = "EURUSD=X"  #EURUSD=X
INTERVAL         = "1h"         # 1-hour bars
EMA_LEN          = 200
BACKCANDLES_PREV = 5     # "previous 5 + current" => window of 6
RR               = 1     # TP = RR * SL
SW_WINDOW        = 4     # swing window size: previous 5 + current = 6 bars

# about ~1 year of 1h data
END   = datetime.utcnow().date().isoformat()
START = (datetime.utcnow() - timedelta(days=400)).date().isoformat()

In [83]:
def fetch_data(symbol: str, start: str, end: str, interval: str) -> pd.DataFrame:
    df = yf.download(symbol, start=start, end=end, interval=interval,
                     auto_adjust=True, progress=False, threads=False)

    if df.empty:
        raise ValueError(f"No data returned for {symbol} @ {interval}. "
                         "Try a different symbol/interval or earlier START.")

    # Handle new yfinance MultiIndex format (Price, Ticker)
    if isinstance(df.columns, pd.MultiIndex):
        try:
            df = df.xs(symbol, axis=1, level=1)  # Keep only this ticker’s data
        except KeyError:
            possible = [lev for lev in df.columns.levels[1]]
            raise KeyError(f"Symbol '{symbol}' not found in MultiIndex columns. "
                           f"Available: {possible}")
    else:
        pass

    # Ensure standardized column names
    df.columns = [c.title() for c in df.columns]
    return df.dropna()

In [84]:
# EMA trend signal
def ema_trend_signal(df: pd.DataFrame, i: int, backcandles_prev: int) -> int:
    """
    Return:
      1  if ALL of [i-backcandles_prev .. i] have Open>EMA and Close>EMA  (uptrend)
     -1  if ALL of [i-backcandles_prev .. i] have Open<EMA and Close<EMA  (downtrend)
      0  otherwise
    Uses only current and past candles.
    """
    if i < backcandles_prev:
        return 0
    if np.isnan(df["EMA200"].iloc[i]):
        return 0

    start = i - backcandles_prev
    seg = df.iloc[start:i+1]
    up   = ((seg["Open"] > seg["EMA200"]) & (seg["Close"] > seg["EMA200"])).all()
    down = ((seg["Open"] < seg["EMA200"]) & (seg["Close"] < seg["EMA200"])).all()
    return 1 if up else (-1 if down else 0)

In [90]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Indicators
    out["EMA200"] = ta.ema(out["Close"], length=EMA_LEN)
    macd = ta.macd(out["Close"], fast=12, slow=26, signal=9)
    out = out.join(macd)

    # ema_signal via the function specified
    out["ema_signal"] = 0
    out["ema_signal"] = [ema_trend_signal(out, i, BACKCANDLES_PREV) for i in range(len(out))]

    # MACD cross logic (current & previous bar only)
    macd_line = out["MACD_12_26_9"]
    macd_sig  = out["MACDs_12_26_9"]
    macd_hist = out["MACDh_12_26_9"]
    macd_line_prev = macd_line.shift(1)
    macd_sig_prev  = macd_sig.shift(1)
    macd_hist_prev = macd_hist.shift(1)

    # # Long when MACD crosses up while both are BELOW 0
    # bull_cross_below0 = (
    #     (macd_line_prev <= macd_sig_prev) &
    #     (macd_line > macd_sig) &
    #     (macd_line < 0) & (macd_sig < 0)
    # )
    # # Short when MACD crosses down while both are ABOVE 0
    # bear_cross_above0 = (
    #     (macd_line_prev >= macd_sig_prev) &
    #     (macd_line < macd_sig) &
    #     (macd_line > 0) & (macd_sig > 0)
    # )

    # bull_cross_below0 = ( (macd_sig_prev >= 0) 
    #                      & (macd_sig < 0) ) 
    #                     # & (macd_line <= 0) )
    
    # bear_cross_above0 = ( (macd_sig_prev <= 0) 
    #                      & (macd_sig > 0) )
    #                      # & (macd_line >=0) )


    # Params
    hist_thresh = 4e-6
    hist_window = 7  # current bar + last 6
    # "Any of last 7 bars" condition via rolling extremes
    hist_below_win = macd_hist.rolling(hist_window, min_periods=hist_window).min() < -hist_thresh
    hist_above_win = macd_hist.rolling(hist_window, min_periods=hist_window).max() >  hist_thresh

    # --- Entries ---
    # Long: MACD line crosses ABOVE signal line while both are BELOW zero (bullish resumption)
    bull_cross_below0 = (
        hist_below_win &                          # pullback was deep enough in last 7 bars
        (macd_line_prev <= macd_sig_prev) &       # was not above on prior bar
        (macd_line > macd_sig) &                  # crosses above now
        (macd_line < 0) & (macd_sig < 0)          # cross happens below zero
    )

    # Short: MACD line crosses BELOW signal line while both are ABOVE zero (bearish resumption)
    bear_cross_above0 = (
        hist_above_win &                          # pullback was deep enough in last 7 bars
        (macd_line_prev >= macd_sig_prev) &       # was not below on prior bar
        (macd_line < macd_sig) &                  # crosses below now
        (macd_line > 0) & (macd_sig > 0)          # cross happens above zero
    )

    out["MACD_signal"] = 0
    out.loc[bull_cross_below0, "MACD_signal"] = 1
    out.loc[bear_cross_above0, "MACD_signal"] = -1

    # Precomputed combined signal
    out["pre_signal"] = 0
    out.loc[(out["ema_signal"] == 1) & (out["MACD_signal"] == 1), "pre_signal"] = 1
    out.loc[(out["ema_signal"] == -1) & (out["MACD_signal"] == -1), "pre_signal"] = -1
    
    # Drop early NaNs from indicators
    out = out.dropna().copy()
    return out


In [91]:
raw = fetch_data(SYMBOL, START, END, INTERVAL)
df  = build_features(raw)
df[df['pre_signal']!=0]

,Close,High,Low,Open,Volume,EMA200,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,ema_signal,MACD_signal,pre_signal
Datetime,,,,,,,,,,,,
2024-07-25 21:00:00+00:00,1.085069,1.085069,1.084011,1.084952,0,1.088643,0.000088,-0.000010,0.000099,-1,-1,-1
2024-07-26 07:00:00+00:00,1.085187,1.085776,1.084834,1.085541,0,1.088361,0.000165,-0.000023,0.000188,-1,-1,-1
2024-07-26 19:00:00+00:00,1.086130,1.086248,1.085894,1.085894,0,1.088114,0.000249,-0.000010,0.000259,-1,-1,-1
2024-07-29 04:00:00+00:00,1.086248,1.086602,1.086130,1.086484,0,1.087976,0.000218,-0.000005,0.000224,-1,-1,-1
2024-07-31 16:00:00+00:00,1.081198,1.082485,1.080847,1.082485,0,1.085528,0.000091,-0.000087,0.000178,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-02 17:00:00+00:00,1.179941,1.180220,1.179245,1.179245,0,1.169193,-0.000172,0.000138,-0.000310,1,1,1
2025-07-03 23:00:00+00:00,1.177440,1.177440,1.175917,1.176332,0,1.171553,-0.000812,0.000012,-0.000824,1,1,1
2025-07-25 18:00:00+00:00,1.174950,1.174950,1.173985,1.173985,0,1.171176,-0.000506,0.000117,-0.000624,1,1,1


In [ ]:
s = df['pre_signal'].astype(int)
# Last *non-zero* signal before the current row (ignores zeros), shifted so it's truly "prior"
prev_nz = s.replace(0, np.nan).ffill().shift(1)

# First opposite signal after a regime change: current non-zero AND opposite to last non-zero prior
flip_first = (s != 0) & prev_nz.notna() & (s != prev_nz)

# Zero out only that first opposite signal
df.loc[flip_first, 'pre_signal'] = 0
len(df[df['pre_signal']!=0])

58

### Strategies with different exit approaches

In [88]:
# Strategy stop loss moved to entry price at break even level (profit), initial SL at swing low/high

class MACDEMA_SwingWindow_BE(Strategy):
    rr        = RR
    sw_window = SW_WINDOW  # number of previous bars (excluding current) to include

    def init(self):
        self._moved_be = False
        self._init_sl_dist = None  # discovered from the actual Trade after entry

    def _move_to_breakeven_if_ready(self):
        if not self.trades:
            return
        tr = self.trades[0]

        # derive initial SL distance from actual opened trade
        if self._init_sl_dist is None and tr.sl is not None:
            self._init_sl_dist = abs(tr.entry_price - tr.sl)

        if not self._init_sl_dist or self._moved_be:
            return

        price = float(self.data.Close[-1])
        if tr.is_long and (price - tr.entry_price) >= self._init_sl_dist:
            tr.sl = tr.entry_price
            self._moved_be = True
        elif tr.is_short and (tr.entry_price - price) >= self._init_sl_dist:
            tr.sl = tr.entry_price
            self._moved_be = True

    def _enough_history(self) -> bool:
        # need current + sw_window previous bars
        return len(self.data.Close) >= (self.sw_window + 1)

    def next(self):
        close = float(self.data.Close[-1])
        sig   = int(self.data.df["pre_signal"].iloc[-1])

        if not self.position:
            self._moved_be = False
            self._init_sl_dist = None

            if not self._enough_history():
                return

            lows  = self.data.df["Low"].iloc[-(self.sw_window + 1):]
            highs = self.data.df["High"].iloc[-(self.sw_window + 1):]

            if sig == 1:  # Long: SL at min(low) over [current .. current-5]
                sw_low = float(np.min(lows))
                if not np.isfinite(sw_low) or sw_low >= close:
                    return  # invalid stop (would be above/at entry)
                dist = close - sw_low
                sl = sw_low
                tp = close + self.rr * dist
                self.buy(sl=sl, tp=tp)

            elif sig == -1:  # Short: SL at max(high) over [current .. current-5]
                sw_high = float(np.max(highs))
                if not np.isfinite(sw_high) or sw_high <= close:
                    return  # invalid stop (would be below/at entry)
                dist = sw_high - close
                sl = sw_high
                tp = close - self.rr * dist
                self.sell(sl=sl, tp=tp)

        else:
            self._move_to_breakeven_if_ready()

In [46]:
# normal no break even change SL
class MACDEMA_SwingWindow(Strategy):
    rr        = RR          # e.g., 2.0
    sw_window = SW_WINDOW   # e.g., 5 → current + prev 5 bars

    def init(self):
        pass  # no state needed; exits handled purely by SL/TP

    def _enough_history(self) -> bool:
        return len(self.data.Close) >= (self.sw_window + 1)

    def next(self):
        close = float(self.data.Close[-1])
        sig   = int(self.data.df["pre_signal"].iloc[-1])

        # Only one position at a time (with exclusive_orders=True)
        if self.position:
            return

        if not self._enough_history():
            return

        lows  = self.data.df["Low"].iloc[-(self.sw_window + 1):]
        highs = self.data.df["High"].iloc[-(self.sw_window + 1):]

        if sig == 1:
            # Long: SL = min(low) over window; TP by RR
            sw_low = float(np.min(lows))
            if not np.isfinite(sw_low) or sw_low >= close:
                return  # invalid stop; skip
            dist = close - sw_low
            self.buy(sl=sw_low, tp=close + self.rr * dist)

        elif sig == -1:
            # Short: SL = max(high) over window; TP by RR
            sw_high = float(np.max(highs))
            if not np.isfinite(sw_high) or sw_high <= close:
                return
            dist = sw_high - close
            self.sell(sl=sw_high, tp=close - self.rr * dist)

In [47]:
class MACDEMA_SwingOrATR(Strategy):
    rr: float        = 2.0     # TP multiple
    sw_window: int   = 5       # uses current + prev sw_window bars
    atr_mult: float  = 2.0     # ATR factor for floor on SL distance
    atr_len: int     = 14      # ATR length if we need to compute it

    def init(self):
        # Ensure ATR column exists (name: 'ATR')
        if "ATR" not in self.data.df.columns:
            # Lazy compute ATR once; avoids lookahead since we use current bar only
            import pandas_ta as ta
            df = self.data.df
            df["ATR"] = ta.atr(df["High"], df["Low"], df["Close"], length=self.atr_len)

    def _enough_history(self) -> bool:
        # need at least current + sw_window previous bars
        return len(self.data.Close) >= (self.sw_window + 1)

    def next(self):
        close = float(self.data.Close[-1])
        sig   = int(self.data.df["pre_signal"].iloc[-1])

        if self.position:
            return
        if not self._enough_history():
            return

        # Swing extremes over window [current .. current - sw_window]
        lows  = self.data.df["Low"].iloc[-(self.sw_window + 1):]
        highs = self.data.df["High"].iloc[-(self.sw_window + 1):]
        atr   = float(self.data.df["ATR"].iloc[-1]) if "ATR" in self.data.df.columns else float("nan")

        # ATR-based distance floor
        atr_dist = self.atr_mult * atr if np.isfinite(atr) else 0.0

        if sig == 1:
            sw_low = float(np.min(lows))
            if not np.isfinite(sw_low) or sw_low >= close:
                return  # invalid swing stop
            swing_dist = close - sw_low
            sl_dist = max(swing_dist, atr_dist)
            if sl_dist <= 0:
                return
            self.buy(sl=close - sl_dist, tp=close + self.rr * sl_dist)

        elif sig == -1:
            sw_high = float(np.max(highs))
            if not np.isfinite(sw_high) or sw_high <= close:
                return
            swing_dist = sw_high - close
            sl_dist = max(swing_dist, atr_dist)
            if sl_dist <= 0:
                return
            self.sell(sl=close + sl_dist, tp=close - self.rr * sl_dist)


In [48]:
class MACDEMA_ATRorBand(Strategy):
    rr: float        = 2.0       # TP multiple
    atr_mult: float  = 2.0       # ATR coefficient
    atr_len: int     = 14        # ATR length if we need to compute it
    band_pct: float  = 0.01      # 2% band around current high/low

    def init(self):
        # Ensure ATR exists (column name: 'ATR')
        if "ATR" not in self.data.df.columns:
            import pandas_ta as ta
            df = self.data.df
            df["ATR"] = ta.atr(df["High"], df["Low"], df["Close"], length=self.atr_len)

    def next(self):
        # one position at a time
        if self.position:
            return

        close = float(self.data.Close[-1])
        high  = float(self.data.df["High"].iloc[-1])
        low   = float(self.data.df["Low"].iloc[-1])
        sig   = int(self.data.df["pre_signal"].iloc[-1])

        atr = float(self.data.df["ATR"].iloc[-1]) if "ATR" in self.data.df.columns else float("nan")
        atr_floor = self.atr_mult * atr if np.isfinite(atr) else 0.0

        if sig == 1:
            # Long: SL candidate at low * (1 - band_pct)
            band_sl_price = low * (1.0 - self.band_pct)
            band_dist = max(0.0, close - band_sl_price)
            sl_dist = max(band_dist, atr_floor)
            if sl_dist <= 0:
                return
            self.buy(sl=close - sl_dist, tp=close + self.rr * sl_dist)

        elif sig == -1:
            # Short: SL candidate at high * (1 + band_pct)
            band_sl_price = high * (1.0 + self.band_pct)
            band_dist = max(0.0, band_sl_price - close)
            sl_dist = max(band_dist, atr_floor)
            if sl_dist <= 0:
                return
            self.sell(sl=close + sl_dist, tp=close - self.rr * sl_dist)


In [102]:
class MACDEMA_ATRTrail(Strategy):
    """
    Enters on df['pre_signal'] (1 = long, -1 = short) and manages the position
    with a pure ATR-based trailing stop (no fixed TP).

    - Initial stop distance = ATR * atr_mult
    - Long: trail at (highest high since entry - trail_dist), only ratchet upward
    - Short: trail at (lowest low since entry + trail_dist), only ratchet downward
    """
    atr_mult: float = 2.0   # <-- optimize this later
    atr_len:  int   = 14    # ATR length if not already in df as 'ATR'
    rr: float        = 2.0       # dummy

    def init(self):
        df = self.data.df
        # Ensure ATR exists as a column named 'ATR'
        if "ATR" not in df.columns:
            import pandas_ta as ta
            df["ATR"] = ta.atr(df["High"], df["Low"], df["Close"], length=self.atr_len)

        # trailing state (reset on each (new) entry / when flat)
        self._trail_dist = None
        self._peak = None      # highest high since entry (for long)
        self._trough = None    # lowest low since entry (for short)

    def _reset_trailing_state(self):
        self._trail_dist = None
        self._peak = None
        self._trough = None

    def next(self):
        df = self.data.df
        close = float(self.data.Close[-1])
        high  = float(df["High"].iloc[-1])
        low   = float(df["Low"].iloc[-1])
        sig   = int(df["pre_signal"].iloc[-1])

        atr = float(df["ATR"].iloc[-1]) if "ATR" in df.columns else float("nan")

        # If we have an open position, update the trailing stop
        if self.position:
            tr = self.trades[0]

            # initialize trail_dist if needed (defensive)
            if (self._trail_dist is None) or not np.isfinite(self._trail_dist):
                self._trail_dist = (self.atr_mult * atr) if np.isfinite(atr) else 0.0

            if tr.is_long:
                # Update peak and trail SL upward only
                self._peak = high if self._peak is None else max(self._peak, high)
                new_sl = self._peak - self._trail_dist
                if tr.sl is None:
                    tr.sl = new_sl
                else:
                    tr.sl = max(tr.sl, new_sl)  # tighten only (never loosen)
            else:
                # Update trough and trail SL downward only
                self._trough = low if self._trough is None else min(self._trough, low)
                new_sl = self._trough + self._trail_dist
                if tr.sl is None:
                    tr.sl = new_sl
                else:
                    tr.sl = min(tr.sl, new_sl)  # tighten only (never loosen)
            return  # Don't open new positions while one is active

        # Flat → consider a new entry
        self._reset_trailing_state()

        if not np.isfinite(atr) or atr <= 0:
            return  # no ATR → no trade

        trail = self.atr_mult * atr

        if sig == 1:
            # Long entry with initial ATR-based stop
            self._trail_dist = trail
            self._peak = high  # initialize peak at entry bar's high
            self.buy(sl=close - trail)

        elif sig == -1:
            # Short entry with initial ATR-based stop
            self._trail_dist = trail
            self._trough = low  # initialize trough at entry bar's low
            self.sell(size=0.99, sl=close + trail)

In [103]:
import matplotlib.pyplot as plt

RR_GRID          = [1.0, 1.5, 2.0, 2.5, 3.0]
SW_GRID          = list(range(3, 9))   # window = current+prev SW bars; try 3..8
def optimize_and_heatmap(df: pd.DataFrame, strategy_name:object):
    bt = Backtest(
        df,
        strategy_name,
        cash=100_000,
        commission=0.0002,
        trade_on_close=False,   # next-bar execution → no lookahead
        hedging=False,
        exclusive_orders=False
    )

    # Optimize rr and sw_window; maximize total Return [%]
    stats, heatmap = bt.optimize(
        rr=RR_GRID,
        sw_window=SW_GRID,
        maximize="Return [%]",
        return_heatmap=True,
        constraint=lambda p: p.rr > 0 and p.sw_window >= 1
    )

    print("Best params:")
    print(stats._strategy)
    print(stats)
    print(heatmap)

In [96]:
# Strategies names: MACDEMA_SwingWindow_BE, MACDEMA_SwingWindow, MACDEMA_SwingOrATR, MACDEMA_ATRorBand
print(f"Fetching {SYMBOL} from {START} to {END} @ {INTERVAL} …")
raw = fetch_data(SYMBOL, START, END, INTERVAL)
df  = build_features(raw)
optimize_and_heatmap(df, MACDEMA_SwingOrATR)

Fetching EURUSD=X from 2024-07-13 to 2025-08-17 @ 1h …


Best params:
MACDEMA_SwingOrATR(rr=2.5,sw_window=8)
Start                     2024-07-25 07:00...
End                       2025-08-15 21:00...
Duration                    386 days 14:00:00
Exposure Time [%]                   17.773706
Equity Final [$]                102060.610724
Equity Peak [$]                 105154.059213
Return [%]                           2.060611
Buy & Hold Return [%]                 7.87964
Return (Ann.) [%]                    1.742299
Volatility (Ann.) [%]                3.500303
Sharpe Ratio                         0.497756
Sortino Ratio                        0.856482
Calmar Ratio                         0.437496
Max. Drawdown [%]                    -3.98243
Avg. Drawdown [%]                   -0.362034
Max. Drawdown Duration      141 days 18:00:00
Avg. Drawdown Duration        9 days 04:00:00
# Trades                                   48
Win Rate [%]                        39.583333
Best Trade [%]                       1.944969
Worst Trade [%]             

In [107]:
bt = Backtest(
    df,
    MACDEMA_ATRTrail,
    cash=100_000,
    commission=0.0002,
    trade_on_close=False,   # next-bar execution → get away from lookahead
    hedging=False,
    exclusive_orders=False,
    margin=1/5
    )
bt.run()

Start                     2024-07-25 07:00...
End                       2025-08-15 21:00...
Duration                    386 days 14:00:00
Exposure Time [%]                    9.100626
Equity Final [$]                 94519.933576
Equity Peak [$]                 102760.657332
Return [%]                          -5.480066
Buy & Hold Return [%]                 7.87964
Return (Ann.) [%]                   -4.149594
Volatility (Ann.) [%]                9.549917
Sharpe Ratio                              0.0
Sortino Ratio                             0.0
Calmar Ratio                              0.0
Max. Drawdown [%]                   -8.933134
Avg. Drawdown [%]                   -4.581224
Max. Drawdown Duration      141 days 05:00:00
Avg. Drawdown Duration       77 days 05:00:00
# Trades                                   62
Win Rate [%]                        30.645161
Best Trade [%]                       1.389903
Worst Trade [%]                     -0.387018
Avg. Trade [%]                    

In [108]:
stats = bt.optimize(
    atr_mult = list(np.arange(1.0, 4.25, 0.25)),
    maximize = 'Return [%]',  # or 'Equity Final [$]', 'SQN', etc.
)
stats

Start                     2024-07-25 07:00...
End                       2025-08-15 21:00...
Duration                    386 days 14:00:00
Exposure Time [%]                   24.186899
Equity Final [$]                134429.316381
Equity Peak [$]                 157880.265606
Return [%]                          34.429316
Buy & Hold Return [%]                 7.87964
Return (Ann.) [%]                   27.627492
Volatility (Ann.) [%]               22.521167
Sharpe Ratio                         1.226734
Sortino Ratio                        2.999434
Calmar Ratio                         1.635502
Max. Drawdown [%]                  -16.892363
Avg. Drawdown [%]                   -1.359356
Max. Drawdown Duration      136 days 08:00:00
Avg. Drawdown Duration        6 days 16:00:00
# Trades                                   47
Win Rate [%]                        42.553191
Best Trade [%]                       3.003436
Worst Trade [%]                     -0.949797
Avg. Trade [%]                    

In [106]:
stats._strategy

<Strategy MACDEMA_ATRTrail(atr_mult=3.75)>

In [58]:
bt.plot()

f:\Python\Lib\site-packages\backtesting\_plotting.py:250: BokehDeprecationWarning:

Passing lists of formats for DatetimeTickFormatter scales was deprecated in Bokeh 3.0. Configure a single string format for each scale

f:\Python\Lib\site-packages\backtesting\_plotting.py:250: BokehDeprecationWarning:

Passing lists of formats for DatetimeTickFormatter scales was deprecated in Bokeh 3.0. Configure a single string format for each scale



GridPlot(id='p1289', ...)

In [93]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_ema_macd(
    df: pd.DataFrame,
    ema_col: str = "EMA200",
    signal_col: str = "pre_signal",
    macd_col: str = "MACD_12_26_9",
    macds_col: str = "MACDs_12_26_9",
    macdh_col: str = "MACDh_12_26_9",
    title: str | None = None,
    start: str | None = None,
    end: str | None = None,
):
    # Checks
    required_ohlc = {"Open", "High", "Low", "Close"}
    missing = required_ohlc - set(df.columns)
    if missing:
        raise ValueError(f"Missing OHLC columns: {missing}")
    for col in [ema_col, signal_col, macd_col, macds_col, macdh_col]:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in df.")

    # Slice
    _df = df.loc[start:end].copy() if (start or end) else df.copy()
    if _df.empty:
        raise ValueError("Selected slice is empty. Check 'start'/'end'.")

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        row_heights=[0.68, 0.32],
        subplot_titles=(title or "Price / EMA / Signals", "MACD")
    )

    # Row 1: Candles
    fig.add_trace(
        go.Candlestick(
            x=_df.index, open=_df["Open"], high=_df["High"],
            low=_df["Low"], close=_df["Close"],
            name="Candles", showlegend=False
        ),
        row=1, col=1
    )

    # EMA
    fig.add_trace(
        go.Scatter(x=_df.index, y=_df[ema_col], mode="lines", name=ema_col),
        row=1, col=1
    )

    # Signals → purple triangles
    sig = _df[signal_col].fillna(0)
    hl_range = (_df["High"] - _df["Low"]).astype(float)
    # Use 'where' to substitute 0 ranges with a tiny fraction of price
    hl_range = hl_range.where(hl_range != 0, _df["Close"] * 0.001)
    offset = np.maximum(hl_range * 0.2, _df["Close"] * 0.0005)

    long_mask = sig.eq(1)
    if long_mask.any():
        long_y = (_df["Low"] - offset)[long_mask]
        fig.add_trace(
            go.Scatter(
                x=_df.index[long_mask], y=long_y,
                mode="markers",
                marker=dict(symbol="triangle-up", size=15, color="green"),
                name="Long (+1)"
            ),
            row=1, col=1
        )

    short_mask = sig.eq(-1)
    if short_mask.any():
        short_y = (_df["High"] + offset)[short_mask]
        fig.add_trace(
            go.Scatter(
                x=_df.index[short_mask], y=short_y,
                mode="markers",
                marker=dict(symbol="triangle-down", size=15, color="red"),
                name="Short (-1)"
            ),
            row=1, col=1
        )

    # Row 2: MACD
    fig.add_trace(go.Bar(x=_df.index, y=_df[macdh_col], name="MACD Hist", opacity=0.5), row=2, col=1)
    fig.add_trace(go.Scatter(x=_df.index, y=np.zeros(len(_df)), mode="lines",
                             line=dict(width=1, dash="dash"), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=_df.index, y=_df[macd_col],  mode="lines", name="MACD"),  row=2, col=1)
    fig.add_trace(go.Scatter(x=_df.index, y=_df[macds_col], mode="lines", name="Signal"), row=2, col=1)

    fig.update_layout(
        title=title or "EMA & Signals with MACD",
        xaxis_rangeslider_visible=False,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=20, t=70, b=40),
        height=800
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="MACD",  row=2, col=1)
    return fig


fig = plot_ema_macd(
    df[1000:1500],
    ema_col="EMA200",
    signal_col="pre_signal",
    macd_col="MACD_12_26_9",
    macds_col="MACDs_12_26_9",
    macdh_col="MACDh_12_26_9",
    title="EMA + Trade Signals with MACD"
)
fig.show()

In [ ]:
df.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'EMA200', 'MACD_12_26_9',
       'MACDh_12_26_9', 'MACDs_12_26_9', 'ema_signal', 'MACD_signal',
       'pre_signal'],
      dtype='object')

In [ ]:
df

,Close,High,Low,Open,Volume,EMA200,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,ema_signal,MACD_signal,pre_signal
2024-07-18 07:00:00+00:00,1.093374,1.093733,1.092896,1.093374,0,1.087234,0.000440,-0.000196,0.000636,0,0,0
2024-07-18 08:00:00+00:00,1.093374,1.094092,1.093255,1.093374,0,1.087295,0.000383,-0.000203,0.000585,0,0,0
2024-07-18 09:00:00+00:00,1.093374,1.093494,1.093135,1.093255,0,1.087356,0.000333,-0.000202,0.000535,0,0,0
2024-07-18 10:00:00+00:00,1.093613,1.093972,1.093255,1.093255,0,1.087418,0.000310,-0.000180,0.000490,0,0,0
2024-07-18 11:00:00+00:00,1.093016,1.093613,1.092896,1.093613,0,1.087474,0.000240,-0.000200,0.000440,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-11 18:00:00+00:00,1.161575,1.161710,1.161170,1.161440,0,1.161521,-0.001201,-0.000402,-0.000799,0,0,0
2025-08-11 19:00:00+00:00,1.160766,1.161575,1.160631,1.161440,0,1.161513,-0.001259,-0.000369,-0.000891,0,0,0
2025-08-11 20:00:00+00:00,1.161980,1.162250,1.160766,1.160766,0,1.161518,-0.001194,-0.000243,-0.000951,0,0,0
2025-08-11 21:00:00+00:00,1.161845,1.162250,1.161440,1.162250,0,1.161521,-0.001140,-0.000151,-0.000989,0,0,0
